# Ingestion Pipeline Experiments — docs → pgvector

Experimentation notebook for **module 1.4** (`app/retrieval/ingestion.py`). The production
pipeline is a **separate offline job** — it is not part of the request graph. Use this
notebook to settle the ingestion decisions (loaders, chunking parameters, embedding model,
collection writes), then port the final choices into `app/retrieval/ingestion.py` +
`app/config/settings.py` and verify via the Stage 1 cells of `v1_module_tests.ipynb`.

**Design constraints (from the v1 reference, do not deviate):**
- Vector store: **pgvector** (reuses existing PostgreSQL — no new infra)
- **One collection per product**, names from the registry (`ProductConfig.doc_collection`)
- Chunks carry **source metadata** (needed for `[D1]`-style citations downstream)
- Registry-driven: no hardcoded product names

**LangChain components used:** document loaders (`PyPDFLoader` / `Docx2txtLoader` /
`TextLoader`), `RecursiveCharacterTextSplitter`, `OpenAIEmbeddings`, `PGVector`
(`langchain_postgres`).

---
## 0. Setup

In [2]:
import sys, os
from pathlib import Path

# Walk up from the CWD to the repo root, identified by a marker file. Depth-independent,
# so this cell is correct from notebooks/, notebooks/experiments/, or the repo root.
def _find_repo_root(start: Path) -> Path:
    for d in (start, *start.parents):
        if (d / "pyproject.toml").exists():
            return d
    raise RuntimeError(f"repo root (pyproject.toml) not found above {start}")

REPO_ROOT = _find_repo_root(Path.cwd())
sys.path.insert(0, str(REPO_ROOT))

from dotenv import load_dotenv
load_dotenv()

assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY not set"
PG_CONN = os.environ["DATABASE_URL"]    # postgresql+psycopg://... (see .env)
print("Repo root:", REPO_ROOT)

Repo root: c:\Users\DELL\OneDrive - scoptanalytics.com\Documents\GitHub\finspring_ai_chatbot


In [3]:
# Registry drives everything: products and their collection names
from app.core.registry import Registry

registry = Registry(path=str(REPO_ROOT / "app/config/products.yaml"))
for pid in registry.product_ids():
    print(pid, "->", registry.get(pid).doc_collection)

digital_fd -> fd_docs
digital_gold -> gold_docs
bonds -> bond_docs
mutual_funds -> mf_docs


---
## 1. Document inventory

Convention: raw product documentation lives at `data/docs/<product_id>/`, one folder per
product, matching registry IDs. Supported formats: `.pdf`, `.docx`, `.md`, `.txt`.

In [4]:
DOCS_ROOT = REPO_ROOT / "data" / "docs"
DOC_SUFFIXES = {".pdf", ".docx", ".md", ".txt"}

def is_ingestable(p: Path) -> bool:
    """Real product documents only. Excludes Word/Office lock files ('~$name.docx'):
    they match .docx but are not documents, and Docx2txtLoader fails on them.
    A KB file left open in Word would otherwise break ingestion with a confusing error."""
    return (p.is_file()
            and p.suffix.lower() in DOC_SUFFIXES
            and not p.name.startswith("~$"))

inventory = {}
for pid in registry.product_ids():
    folder = DOCS_ROOT / pid
    files = sorted(p for p in folder.glob("*") if is_ingestable(p)) if folder.exists() else []
    inventory[pid] = files
    print(f"{pid}: {len(files)} file(s)")
    for f in files:
        print("   ", f.name)
    # Surface what was passed over, so a skipped file is never silent.
    if folder.exists():
        ignored = [p.name for p in folder.glob("*")
                   if p.is_file() and not is_ingestable(p) and p.name != ".gitkeep"]
        for name in sorted(ignored):
            print("    (ignored)", name)

# Non-fatal by design: this is the inventory cell, and experimenting on a subset of products
# is legitimate. Downstream cells skip products with no docs. The hard failure belongs in
# ingestion.py, where a run that finds no documents at all is a broken job, not a choice.
missing = [pid for pid, fs in inventory.items() if not fs]
if missing:
    print("\nWARNING — no docs found for:", missing)

digital_fd: 1 file(s)
    digital_fixed_deposits_kb.docx
digital_gold: 1 file(s)
    digital_gold_kb.docx
bonds: 1 file(s)
    bonds_kb.docx
mutual_funds: 1 file(s)
    mutual_funds_kb.docx


---
## 2. Loading — LangChain document loaders

One loader per file type. Every loaded `Document` gets `product` and `source` metadata —
`source` is what the retriever surfaces and what citations ultimately point back to.

**Chunk-metadata contract — two groups, stamped at different stages:**

*Provenance* (stamped here, at load time — describes the source file):
`product`, `source`, `doc_version`, `ingested_at`, `content_hash`.
Purpose: answer → exact document version audit trail.

*Build config* (stamped in §4, at chunk time — describes how the vector was produced):
`embedding_model`, `chunk_size`, `chunk_overlap`.
Purpose: let the retriever detect that `settings.py` has drifted from what the store was
actually built with. An `embedding_model` mismatch is otherwise **silent** — query vectors
from model B compared against document vectors from model A return arbitrary chunks with
no error, and every downstream gate (`verify` included) still passes.

In [6]:
from langchain_community.document_loaders import PyPDFLoader, Docx2txtLoader, TextLoader
import hashlib, re
from datetime import datetime, timezone

LOADERS = {
    ".pdf":  PyPDFLoader,
    ".docx": Docx2txtLoader,
    ".md":   TextLoader,
    ".txt":  TextLoader,
}

# Provenance half of the contract — stamped at load time, describes the source file.
# The build-config half (embedding_model, chunk_size, chunk_overlap) is stamped in §4.
PROVENANCE_META = {"product", "source", "doc_version", "ingested_at", "content_hash"}

def parse_last_updated(text: str) -> str:
    """Extract the doc's 'Last updated: <Month Year>' header; fall back to 'unknown'."""
    m = re.search(r"Last updated:\**\s*([A-Za-z]+\s+\d{4})", text)
    return m.group(1) if m else "unknown"

def load_product_docs(product_id: str):
    """Load docs and stamp the provenance metadata:
    product, source, doc_version, ingested_at, content_hash.
    Format-agnostic: works identically for .docx, .pdf, .md, .txt because
    doc_version is parsed from the LOADED text, not the raw file bytes."""
    docs = []
    # One timestamp for the whole run, not one per file — every chunk written by a single
    # ingestion carries the same ingested_at, so a run is identifiable in the store.
    # UTC and timezone-aware: the offline job may run on a server in another zone, and a
    # naive local timestamp makes the audit trail ambiguous.
    ingested_at = datetime.now(timezone.utc).isoformat(timespec="seconds")
    for path in inventory[product_id]:
        content_hash = hashlib.sha256(path.read_bytes()).hexdigest()[:12]
        loader = LOADERS[path.suffix.lower()](str(path))
        loaded = loader.load()
        # Parse 'Last updated: <Month Year>' from the extracted text — the header just
        # needs to exist in the document body, whatever the file format (Word included).
        full_text = "\n".join(d.page_content for d in loaded)
        doc_version = parse_last_updated(full_text)
        for d in loaded:
            d.metadata["product"] = product_id
            d.metadata["source"] = path.name
            d.metadata["doc_version"] = doc_version           # e.g. "July 2026"
            d.metadata["ingested_at"] = ingested_at           # e.g. "2026-07-30T09:14:22+00:00"
            d.metadata["content_hash"] = content_hash          # ties chunks to exact file version
            docs.append(d)
    return docs

# Smoke test on one product
sample_pid = registry.product_ids()[0]
raw_docs = load_product_docs(sample_pid)
print(f"{sample_pid}: {len(raw_docs)} raw document(s)/page(s)")
print("\nFirst 400 chars of first doc:\n", raw_docs[0].page_content[:400])
print("\nmetadata:", raw_docs[0].metadata)

# Provenance contract check — every doc must carry all five fields
for d in raw_docs:
    missing = PROVENANCE_META - d.metadata.keys()
    assert not missing, f"Missing metadata fields: {missing}"
print("Provenance metadata OK:", sorted(PROVENANCE_META))

# WATCH THIS: 'unknown' means the 'Last updated:' header did not survive text extraction.
# It is not an error here, but it would render as "updated unknown" in every citation
# downstream (respond, Stage 2). Fix the doc or the regex before porting to ingestion.py.
unresolved = {d.metadata["source"] for d in raw_docs if d.metadata["doc_version"] == "unknown"}
if unresolved:
    print("\n!! doc_version unresolved for:", sorted(unresolved))
else:
    print("doc_version parsed:", {d.metadata["source"]: d.metadata["doc_version"] for d in raw_docs})

digital_fd: 1 raw document(s)/page(s)

First 400 chars of first doc:
 Digital Fixed Deposits (FDs) — Product Knowledge Base (India)

Last updated: July 2026 · Jurisdiction: India · For customer-support use. Tax figures are base rates for FY 2026-27 and exclude applicable surcharge and 4% health & education cess. Bank-specific rates, penalties, and features vary; the deposit-taking bank’s terms always prevail.



1. What is a Fixed Deposit?

A Fixed Deposit (FD), als

metadata: {'source': 'digital_fixed_deposits_kb.docx', 'product': 'digital_fd', 'doc_version': 'July 2026', 'ingested_at': '2026-07-31T11:28:53+00:00', 'content_hash': '06060ebe0911'}
Provenance metadata OK: ['content_hash', 'doc_version', 'ingested_at', 'product', 'source']
doc_version parsed: {'digital_fixed_deposits_kb.docx': 'July 2026'}


In [7]:
raw_docs

[Document(metadata={'source': 'digital_fixed_deposits_kb.docx', 'product': 'digital_fd', 'doc_version': 'July 2026', 'ingested_at': '2026-07-31T11:28:53+00:00', 'content_hash': '06060ebe0911'}, page_content='Digital Fixed Deposits (FDs) — Product Knowledge Base (India)\n\nLast updated: July 2026 · Jurisdiction: India · For customer-support use. Tax figures are base rates for FY 2026-27 and exclude applicable surcharge and 4% health & education cess. Bank-specific rates, penalties, and features vary; the deposit-taking bank’s terms always prevail.\n\n\n\n1. What is a Fixed Deposit?\n\nA Fixed Deposit (FD), also called a term deposit, is a deposit product where a customer places a lump sum with a bank for a fixed tenure at an interest rate agreed at the time of booking. The rate is locked for the full tenure regardless of subsequent market movements. A Digital FD is the same product booked, managed, and closed entirely online — through a bank’s app, internet banking, or a partner platfor

---
## 3. Chunking experiments — `RecursiveCharacterTextSplitter`

The parameter that most affects retrieval quality. Compare a few configs on real docs,
eyeball the chunks, and pick ONE config to freeze into `settings.py`. Judgment criteria:
chunks should be self-contained enough to answer a question alone (they become the `[D1]`
context units), without being so large that top-k retrieval drags in noise.

In [9]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

CONFIGS = [
    {"chunk_size": 500,  "chunk_overlap": 50},
    {"chunk_size": 1000, "chunk_overlap": 150},
    {"chunk_size": 1500, "chunk_overlap": 200},
]

for cfg in CONFIGS:
    splitter = RecursiveCharacterTextSplitter(**cfg)
    chunks = splitter.split_documents(raw_docs)
    lens = [len(c.page_content) for c in chunks]
    print(f"size={cfg['chunk_size']:>4} overlap={cfg['chunk_overlap']:>3} -> "
          f"{len(chunks):>4} chunks | avg len {sum(lens)//max(len(lens),1)}")

size= 500 overlap= 50 ->   26 chunks | avg len 412
size=1000 overlap=150 ->   13 chunks | avg len 886
size=1500 overlap=200 ->    8 chunks | avg len 1412


In [ ]:
# Inspect actual chunks before deciding. Exploratory only — vary PROBE_* freely.
#
# Binds `_probe_splitter` / `chunks`, deliberately NOT `splitter`: the FREEZE cell below
# owns `splitter`, and rebinding it here would make §4 ingest with exploratory parameters
# while stamp_build_config() still recorded the frozen CHUNK_SIZE/CHUNK_OVERLAP. The
# metadata would then misdescribe the store, and the drift guard could not catch it —
# it compares settings against metadata, and both would agree while the chunks differed.
PROBE_SIZE, PROBE_OVERLAP = 1000, 150
PROBE_SEPARATORS = [r"\n\n\d+\.\s", "\n\n", "\n", " ", ""]   # None -> splitter defaults

_probe_splitter = RecursiveCharacterTextSplitter(
    chunk_size=PROBE_SIZE,
    chunk_overlap=PROBE_OVERLAP,
    **({"separators": PROBE_SEPARATORS, "is_separator_regex": True}
       if PROBE_SEPARATORS else {}),
)
chunks = _probe_splitter.split_documents(raw_docs)
print(f"size={PROBE_SIZE} overlap={PROBE_OVERLAP} "
      f"section_aware={bool(PROBE_SEPARATORS)} -> {len(chunks)} chunks\n")

for c in chunks[:3]:
    print("=" * 80)
    print("source:", c.metadata.get("source"), "| product:", c.metadata.get("product"))
    print(c.page_content[:500])

In [52]:
from termcolor import COLORS, colored
from random import choice

In [53]:
def shared_overlap_len(prev: str, curr: str, cap: int) -> int:
    """Longest suffix of `prev` that is also a prefix of `curr`, up to `cap` chars.

    Measures the ACTUAL overlap instead of assuming it equals chunk_overlap.
    RecursiveCharacterTextSplitter cuts at separators, so carried-over text is almost
    never exactly chunk_overlap long — and is 0 when the cut lands on a clean boundary,
    since the splitter avoids carrying back a partial paragraph/section.
    """
    for n in range(min(cap, len(prev), len(curr)), 0, -1):
        if prev[-n:] == curr[:n]:
            return n
    return 0


def display_chunks_with_overlaps(chunks, chunk_overlap):
    """Print chunks with their overlap regions highlighted (overlap = white).

    Accepts either a list of str or the Document list that split_documents() returns.
    chunk_overlap is an UPPER BOUND — boundaries reported as 0 shared chars are clean
    section/paragraph breaks, not a failure of the splitter.
    """
    if not chunk_overlap:
        raise ValueError("Chunk Overlap cannot be 0")

    texts = [c.page_content if hasattr(c, "page_content") else c for c in chunks]
    overlap_color = "white"
    colors_list = list(COLORS.keys())[2:8]

    # Real overlap at each boundary i -> i+1, measured not assumed.
    shared = [shared_overlap_len(texts[i], texts[i + 1], chunk_overlap)
              for i in range(len(texts) - 1)]

    print(f"Total Number of Chunks: {len(texts)}")
    if shared:
        zeros = sum(1 for s in shared if s == 0)
        print(f"Requested overlap: {chunk_overlap} | actual shared chars: {shared}")
        print(f"  mean {sum(shared) / len(shared):.0f}, {zeros}/{len(shared)} "
              f"boundaries with zero overlap (clean section/paragraph breaks)\n")

    for num, chunk in enumerate(texts, 1):
        lead = shared[num - 2] if num > 1 else 0             # shared with previous chunk
        tail = shared[num - 1] if num <= len(shared) else 0  # shared with next chunk
        body_end = len(chunk) - tail

        print(f"Chunk {num}: {len(chunk)} chars | leading overlap {lead} | trailing {tail}")
        print(colored(chunk[:lead], overlap_color), end="")
        print(colored(chunk[lead:body_end], choice(colors_list)), end="")
        print(colored(chunk[body_end:], overlap_color), end="\n\n")

In [57]:
chunks[1]

Document(metadata={'source': 'digital_fixed_deposits_kb.docx', 'product': 'digital_fd', 'doc_version': 'July 2026', 'ingested_at': '2026-07-31T11:28:53+00:00', 'content_hash': '06060ebe0911'}, page_content='1. What is a Fixed Deposit?\n\nA Fixed Deposit (FD), also called a term deposit, is a deposit product where a customer places a lump sum with a bank for a fixed tenure at an interest rate agreed at the time of booking. The rate is locked for the full tenure regardless of subsequent market movements. A Digital FD is the same product booked, managed, and closed entirely online — through a bank’s app, internet banking, or a partner platform — with no branch visit or physical paperwork.\n\nKey characteristics: - Fixed, guaranteed interest rate for the chosen tenure. - Tenures typically range from 7 days to 10 years. - Principal and interest are paid by the bank; returns do not depend on market performance. - Regulated by the Reserve Bank of India (RBI) for banks; company/NBFC deposits a

In [55]:
display_chunks_with_overlaps(chunks, 150)

Total Number of Chunks: 16
Requested overlap: 150 | actual shared chars: [0, 0, 124, 0, 0, 0, 0, 0, 0, 90, 0, 0, 0, 0, 0]
  mean 14, 13/15 boundaries with zero overlap (clean section/paragraph breaks)

Chunk 1: 342 chars | leading overlap 0 | trailing 0
Digital Fixed Deposits (FDs) — Product Knowledge Base (India)

Last updated: July 2026 · Jurisdiction: India · For customer-support use. Tax figures are base rates for FY 2026-27 and exclude applicable surcharge and 4% health & education cess. Bank-specific rates, penalties, and features vary; the deposit-taking bank’s terms always prevail.

Chunk 2: 852 chars | leading overlap 0 | trailing 0
1. What is a Fixed Deposit?

A Fixed Deposit (FD), also called a term deposit, is a deposit product where a customer places a lump sum with a bank for a fixed tenure at an interest rate agreed at the time of booking. The rate is locked for the full tenure regardless of subsequent market movements. A Digital FD is the same product booked, managed, a

In [56]:
# FREEZE the decision here once you've compared — this is what goes into settings.py
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 150

# Section-aware separators: the first entry is a REGEX matching numbered headings
# ("\n\n5. Premature withdrawal"), so the splitter cuts at section boundaries first and
# falls back to paragraphs only when a section exceeds CHUNK_SIZE.
# Measured across all four KB docs (default separators -> section-aware):
#   chunks spanning >1 numbered section:   9 -> 1
#   chunks starting at a heading:          6 -> 41
# A heading in the chunk is strong topical signal in the embedding, and gives `generate`
# a titled context unit instead of an orphaned fragment starting mid-sentence.
# Trade-off: short sections become short chunks (min 37 chars on bonds). Sharp embeddings,
# but a tiny chunk still consumes a TOP_K slot — watch this in the §5 retrieval checks.
# Degrades gracefully: a KB with no numbered headings never matches the first separator
# and splits on paragraphs exactly as before.
SPLIT_SEPARATORS = [r"\n\n\d+\.\s", "\n\n", "\n", " ", ""]
SPLIT_IS_REGEX = True

splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=SPLIT_SEPARATORS,
    is_separator_regex=SPLIT_IS_REGEX,
)
print(f"Frozen: chunk_size={CHUNK_SIZE}, chunk_overlap={CHUNK_OVERLAP}, "
      f"section_aware={SPLIT_IS_REGEX}")

# Sanity: chunks should now begin at section headings rather than mid-sentence.
_chunks = splitter.split_documents(raw_docs)
print(f"\n{len(_chunks)} chunks for {sample_pid}; first line of each:")
for i, c in enumerate(_chunks, 1):
    print(f"  [{i:>2}] {len(c.page_content):>4} chars | "
          f"{c.page_content.strip().splitlines()[0][:66]!r}")

Frozen: chunk_size=1000, chunk_overlap=150, section_aware=True

16 chunks for digital_fd; first line of each:
  [ 1]  342 chars | 'Digital Fixed Deposits (FDs) — Product Knowledge Base (India)'
  [ 2]  852 chars | '1. What is a Fixed Deposit?'
  [ 3]  984 chars | '2. Types of Fixed Deposits'
  [ 4]  273 chars | 'Senior-citizen FD: Most banks pay an additional rate (commonly 0.2'
  [ 5]  966 chars | '3. Booking a Digital FD — how it works'
  [ 6]  684 chars | '4. Interest — how it is calculated and paid'
  [ 7]  750 chars | '5. Premature withdrawal (breaking an FD)'
  [ 8]  343 chars | '6. Loan / overdraft against FD'
  [ 9]  910 chars | '7. Safety — deposit insurance (DICGC)'
  [10]  818 chars | '8. Taxation of FD interest (FY 2026-27)'
  [11]  924 chars | 'TDS is deducted on the entire interest once the threshold is cross'
  [12]  505 chars | '9. Common risks and limitations'
  [13]  953 chars | '10. Frequently asked questions'
  [14]  127 chars | 'Is a digital FD different from a bra

---
## 4. Embeddings + pgvector collections

`PGVector` from `langchain_postgres`, one collection per product (name from the registry).
`pre_delete_collection=True` makes a full re-ingest idempotent — re-running replaces the
collection instead of duplicating chunks.

**Build-config stamping (the other half of the metadata contract).** Every chunk also
records `embedding_model`, `chunk_size`, `chunk_overlap` — the settings that produced its
vector. `retriever.py` (module 1.5) reads one chunk per collection at startup and compares
these against `settings.py`, so a config change after ingestion fails loudly instead of
silently degrading retrieval:

- `embedding_model` mismatch → **raise**. Vectors from two different models are not
  comparable; if the dimensions happen to match, pgvector returns arbitrary chunks with no
  error at all. Fail closed.
- `chunk_size` / `chunk_overlap` mismatch → **warn**. The store is merely stale, not wrong.

Either way the fix is the same: re-run the offline job.

In [59]:
from langchain_openai import OpenAIEmbeddings
from langchain_postgres import PGVector

EMBEDDING_MODEL = "text-embedding-3-small"   # freeze into settings.py once validated
embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL)

# Quick sanity: embed one string
vec = embeddings.embed_query("fixed deposit premature withdrawal penalty")
print("Embedding dims:", len(vec))

Embedding dims: 1536


In [60]:
vec

[0.0024433135986328125,
 0.0126495361328125,
 0.036224365234375,
 0.0306243896484375,
 -0.05126953125,
 0.0263214111328125,
 0.043853759765625,
 0.04949951171875,
 0.046966552734375,
 -0.0202484130859375,
 0.06390380859375,
 -0.039886474609375,
 0.009857177734375,
 -0.0576171875,
 0.034210205078125,
 0.0188751220703125,
 -0.007122039794921875,
 0.011688232421875,
 -0.05804443359375,
 0.046875,
 0.01010894775390625,
 0.02325439453125,
 -0.032470703125,
 0.0316162109375,
 -0.0287322998046875,
 -0.0123291015625,
 0.041656494140625,
 -0.0703125,
 0.0282745361328125,
 0.0003457069396972656,
 0.029205322265625,
 -0.0197906494140625,
 0.0273284912109375,
 -0.01509857177734375,
 0.0233154296875,
 0.0293121337890625,
 0.0111846923828125,
 0.046142578125,
 -0.0252227783203125,
 -0.059814453125,
 -0.044769287109375,
 -0.0267486572265625,
 0.00574493408203125,
 -0.018035888671875,
 -0.0294647216796875,
 0.038970947265625,
 0.0477294921875,
 -0.0285186767578125,
 -0.0254058837890625,
 0.04104614257

In [61]:
BUILD_CONFIG_META = {"embedding_model", "chunk_size", "chunk_overlap"}

def stamp_build_config(chunks):
    """Record the settings that produced these vectors, so retriever.py can detect
    that settings.py has drifted from what the store was actually built with."""
    for c in chunks:
        c.metadata["embedding_model"] = EMBEDDING_MODEL
        c.metadata["chunk_size"] = CHUNK_SIZE
        c.metadata["chunk_overlap"] = CHUNK_OVERLAP
    return chunks

def ingest_product(product_id: str) -> int:
    """Load -> chunk -> stamp -> embed -> write to the product's collection. Idempotent."""
    cfg = registry.get(product_id)
    docs = load_product_docs(product_id)
    chunks = stamp_build_config(splitter.split_documents(docs))
    PGVector.from_documents(
        documents=chunks,
        embedding=embeddings,
        collection_name=cfg.doc_collection,      # from the registry — never hardcoded
        connection=PG_CONN,
        pre_delete_collection=True,              # full-replace re-ingest
    )
    return len(chunks)

# Ingest ONE product first
n = ingest_product(sample_pid)
print(f"{sample_pid}: {n} chunks written to '{registry.get(sample_pid).doc_collection}'")

Collection not found


digital_fd: 16 chunks written to 'fd_docs'


In [62]:
# Ingest ALL products (registry-driven loop — this is the shape of ingestion.py's ingest_all)
for pid in registry.product_ids():
    if not inventory[pid]:
        print(f"SKIP {pid} — no docs")
        continue
    n = ingest_product(pid)
    print(f"OK {pid}: {n} chunks -> {registry.get(pid).doc_collection}")

OK digital_fd: 16 chunks -> fd_docs


Collection not found


OK digital_gold: 16 chunks -> gold_docs


Collection not found


OK bonds: 19 chunks -> bond_docs


Collection not found


OK mutual_funds: 13 chunks -> mf_docs


---
## 5. Retrieval sanity checks

Same checks Stage 1 of `v1_module_tests.ipynb` will run against `app/retrieval/retriever.py` —
run them here first against the raw store to validate the *ingestion*, independent of your
retriever code.

In [63]:
def open_store(product_id: str) -> PGVector:
    return PGVector(
        embeddings=embeddings,
        collection_name=registry.get(product_id).doc_collection,
        connection=PG_CONN,
    )

sample_queries = {
    "digital_fd":   "What is the penalty for premature withdrawal of a fixed deposit?",
    "digital_gold": "How is the gold price determined at purchase?",
    "bonds":        "What is the minimum investment amount for bonds?",
    "mutual_funds": "How is NAV calculated?",
}

for pid, q in sample_queries.items():
    if not inventory.get(pid):
        continue
    store = open_store(pid)
    results = store.similarity_search_with_score(q, k=4)
    print(f"\n=== {pid} :: {q}")
    for doc, score in results:
        print(f"  score={score:.3f} [{doc.metadata.get('source')}] {doc.page_content[:110]}...")


=== digital_fd :: What is the penalty for premature withdrawal of a fixed deposit?
  score=0.332 [digital_fixed_deposits_kb.docx] 5. Premature withdrawal (breaking an FD)

Retail FDs (₹1 crore and below) always carry a premature-withdrawal ...
  score=0.467 [digital_fixed_deposits_kb.docx] 2. Types of Fixed Deposits

Cumulative FD: Interest is compounded (typically quarterly) and paid together with...
  score=0.528 [digital_fixed_deposits_kb.docx] Senior-citizen FD: Most banks pay an additional rate (commonly 0.25%–0.75% per annum extra) to depositors aged...
  score=0.533 [digital_fixed_deposits_kb.docx] 1. What is a Fixed Deposit?

A Fixed Deposit (FD), also called a term deposit, is a deposit product where a cu...

=== digital_gold :: How is the gold price determined at purchase?
  score=0.409 [digital_gold_kb.docx] 4. Pricing and charges — complete picture

Live price: Linked to the domestic 24K gold spot price; it moves co...
  score=0.475 [digital_gold_kb.docx] 3. How buying and

In [69]:
# Scoping + full metadata-contract check on RETRIEVED chunks:
# (a) every result in a product's collection carries that product's metadata; Is every chunk that came back actually the same product chunk?
# (b) both halves of the contract survived splitting and storage. Does every chunk still carry all eight metadata fields?
FULL_META = PROVENANCE_META | BUILD_CONFIG_META
for pid in registry.product_ids():
    if not inventory.get(pid):
        continue
    store = open_store(pid)
    results = store.similarity_search("investment", k=8)
    assert all(d.metadata.get("product") == pid for d in results), \
        f"Cross-product contamination in collection {pid}!"
    for d in results:
        missing = FULL_META - d.metadata.keys()
        assert not missing, f"{pid}: chunk lost metadata fields {missing}"
        assert d.metadata["embedding_model"] == EMBEDDING_MODEL, \
            f"{pid}: chunk built with {d.metadata['embedding_model']}, expected {EMBEDDING_MODEL}"
print("OK — collections scoped per product; provenance + build-config intact on every chunk")
print("Sample:", results[0].metadata)

OK — collections scoped per product; provenance + build-config intact on every chunk
Sample: {'source': 'mutual_funds_kb.docx', 'product': 'mutual_funds', 'chunk_size': 1000, 'doc_version': 'July 2026', 'ingested_at': '2026-08-02T16:05:46+00:00', 'content_hash': 'fca122a5b02c', 'chunk_overlap': 150, 'embedding_model': 'text-embedding-3-small'}


In [70]:
sample_pid

'digital_fd'

In [71]:
# Idempotency check: re-ingest one product, chunk count must be stable (no duplication)
n1 = ingest_product(sample_pid)
n2 = ingest_product(sample_pid)
assert n1 == n2, "Re-ingest changed chunk count — pre_delete_collection not working"
store = open_store(sample_pid)
results = store.similarity_search("test", k=200)
assert len(results) == n1, \
    f"Store holds {len(results)} chunks after re-ingest, expected {n1} — duplication"
print(f"OK — idempotent re-ingest ({n1} chunks both runs)")

OK — idempotent re-ingest (16 chunks both runs)


In [72]:
# Config-drift guard — prototype of the check that ports into retriever.py (module 1.5).
# Reads ONE chunk per collection and compares its build config against current settings.
# embedding_model mismatch -> raise (silent corruption); chunk params -> warn (merely stale).
import warnings

class StoreConfigMismatch(RuntimeError):
    """The store was built with different settings than the ones now configured."""

def assert_store_matches(product_id: str, embedding_model: str,
                         chunk_size: int, chunk_overlap: int) -> dict:
    """Compare the store's build config against current settings.

    `embedding_model` is the only field that affects retrieval CORRECTNESS: the query is
    embedded at query time, so a mismatch makes query and document vectors incomparable
    and retrieval returns arbitrary chunks with no error. Hence it raises.

    `chunk_size` / `chunk_overlap` are NOT used at query time — nothing in the retrieval
    path reads them. A mismatch only means the store predates a settings change, so the
    chunks are stale rather than wrong. Hence it warns. Kept because the realistic failure
    is an incomplete edit (settings tuned, ingestion not re-run), and because this is the
    only place the store is compared against the config at all. Costs nothing: the probe
    chunk is already fetched for the embedding check.
    """
    collection = registry.get(product_id).doc_collection
    probe = open_store(product_id).similarity_search("a", k=1)
    if not probe:
        raise StoreConfigMismatch(
            f"collection '{collection}' is empty — run ingestion for '{product_id}'")
    meta = probe[0].metadata
    built_with = meta.get("embedding_model")
    if built_with != embedding_model:
        raise StoreConfigMismatch(
            f"collection '{collection}' was built with embedding model '{built_with}', "
            f"but settings specify '{embedding_model}'. Vectors from different models are "
            f"not comparable — re-run: python -m app.retrieval.ingestion")
    for field, current in (("chunk_size", chunk_size), ("chunk_overlap", chunk_overlap)):
        if meta.get(field) != current:
            warnings.warn(
                f"collection '{collection}' was built with {field}={meta.get(field)}, "
                f"settings say {current}. Store is stale — re-run ingestion.", stacklevel=2)
    return meta

# Happy path: current settings match what we just ingested.
for pid in registry.product_ids():
    if inventory.get(pid):
        assert_store_matches(pid, EMBEDDING_MODEL, CHUNK_SIZE, CHUNK_OVERLAP)
print("OK — every collection matches current settings")

# Negative test: pretend settings.py was switched to -large. Must RAISE, not warn.
# The happy path alone proves nothing — a function with no checks would pass it too.
# This is what proves the guard actually fires.
try:
    assert_store_matches(sample_pid, "text-embedding-3-large", CHUNK_SIZE, CHUNK_OVERLAP)
except StoreConfigMismatch as exc:
    print("\nOK — drift detected as expected:\n ", exc)
else:
    raise AssertionError("drift guard did NOT fire on an embedding-model mismatch")

OK — every collection matches current settings

OK — drift detected as expected:
  collection 'fd_docs' was built with embedding model 'text-embedding-3-small', but settings specify 'text-embedding-3-large'. Vectors from different models are not comparable — re-run: python -m app.retrieval.ingestion


---
## 6. Handoff to production (`app/retrieval/ingestion.py`)

Once the cells above pass and you're satisfied with retrieval quality:

1. Freeze into `app/config/settings.py`: `CHUNK_SIZE`, `CHUNK_OVERLAP`, `EMBEDDING_MODEL`,
   `DATABASE_URL` env name, `DOCS_ROOT` convention (`data/docs/<product_id>/`).
   The notebook-local literals above exist only so you can vary them while experimenting —
   they do **not** come along. `ingestion.py` reads every one of them from settings.
1a. Port the **chunk-metadata contract** as-is, both halves:
   - *provenance* — `product`, `source`, `doc_version`, `ingested_at`, `content_hash`.
     Consumers: `retrieve_docs` carries it into `doc_context`; `respond` renders
     "Source: <source>, updated <doc_version>"; observability logs it with every verify
     verdict (answer → exact doc version audit trail).
   - *build config* — `embedding_model`, `chunk_size`, `chunk_overlap`. Consumer:
     `retriever.py`'s startup drift guard (see step 4).
2. Port `load_product_docs`, `stamp_build_config`, the frozen splitter, and
   `ingest_product` / the registry loop into `app/retrieval/ingestion.py` as
   `ingest_all(registry)` — same code, minus the experimentation cells. Expose it as a
   runnable job (`python -m app.retrieval.ingestion`), since a human or a cron invokes it,
   not the app.
3. Run **Stage 1** of `notebooks/v1_module_tests.ipynb` (cells 1.4–1.6) against the
   production module and mark **1.4** ✅ in `docs/module_tracker.md`.
4. Port `assert_store_matches` into `app/retrieval/retriever.py` (module 1.5), cached
   per collection so it costs one probe per process, not one per query.

Re-run ingestion (offline job) whenever product documentation changes — **or whenever
`EMBEDDING_MODEL`, `CHUNK_SIZE`, or `CHUNK_OVERLAP` changes.** The store is built from
those settings; changing them without re-ingesting leaves the store and the config
disagreeing, which is exactly what the drift guard exists to catch.